# Scrapping Indicators Data

In [ ]:
pip install -r requirements.txt

## Reporteros Sin Fronteras (RSF - Reporters Without Borders) 

In [11]:
import requests
import pandas as pd
import os
from io import StringIO  # Import StringIO

# URL to fetch the request table from
url = "https://rsf.org/en/index?year="

# List of years to iterate over
years = [str(year) for year in range(2002, 2024) if year != 2011]

# Function to get the request table from a given URL
def get_request_table(year):
    year_url = f"{url}{year}"
    response = requests.get(year_url)
    response.raise_for_status()  # Raise an error for bad responses
    
    # Use StringIO to convert the response text to a file-like object
    html_content = StringIO(response.text)
    
    # Get the first table on the page and return it as a DataFrame
    return pd.read_html(html_content)[0]

# Create a DataFrame to hold all the tables for the years
all_data = pd.DataFrame()

# Loop through the years and fetch the tables
for year in years:
    try:
        data = get_request_table(year)
        data['Year'] = year  # Add the year as a column to the data
        all_data = pd.concat([all_data, data], ignore_index=True)
    except Exception as e:
        print(f"Could not retrieve data for {year}: {e}")

# Clean the DataFrame
# 1. Keep only the necessary columns (Country, Year, Global score)
clean_data = all_data[['Countries', 'Year', 'Global score']].copy()

# 2. Rename columns for consistency
clean_data.columns = ['country', 'year', 'wpf_score']

# 3. Normalize the 'wpf_score' to values between 0 and 1
clean_data['wpf_score'] = (clean_data['wpf_score'] - clean_data['wpf_score'].min()) / (clean_data['wpf_score'].max() - clean_data['wpf_score'].min())

# Specify the directory where the CSV should be saved
save_dir = '../data/processed/3_indicators'

# Make sure the directory exists, if not create it
os.makedirs(save_dir, exist_ok=True)

# Save the cleaned data to a CSV file
csv_filename = os.path.join(save_dir, 'wpf_scores.csv')
clean_data.to_csv(csv_filename, index=False)

print(f"Data saved to {csv_filename}")


Data saved to ../data/processed/3_indicators/wpf_scores.csv


In [125]:
# Existing imports and path setup...
import pandas as pd
import json
import os


# Paths
raw_data_path = "../data/raw"
polarisation_path = "../data/reports/polarisation_scores/"
wpf_path = "../data/processed/3_indicators"
output_path = "../data/processed/3_indicators"
os.makedirs(output_path, exist_ok=True)


# === ACLED EVENTS & FATALITIES ===

# Load ISO mapping
with open("iso_country.json") as f:
    iso_mapping = json.load(f)
iso_df = pd.DataFrame.from_dict(iso_mapping, orient='index').reset_index()
iso_df.columns = ['iso_code', 'country']

events = pd.read_csv(os.path.join(raw_data_path, "acled_events.csv"))
fatalities = pd.read_csv(os.path.join(raw_data_path, "acled_fatalities.csv"))

# Rename columns to match
events = events.rename(columns={"Country": "country", "Events": "events"})
fatalities = fatalities.rename(columns={"Country": "country", "Fatalities": "fatalities"})

# Merge with ISO mapping based on country names
events = events.merge(iso_df, on="country", how="left")
fatalities = fatalities.merge(iso_df, on="country", how="left")

# Add year (assuming same year for all rows; change if dynamic)
events["year"] = 2024
fatalities["year"] = 2024

# Merge into a single frame
acled_combined = pd.merge(events, fatalities, on=["country", "iso_code", "year"], how="outer")

# Final columns: country, iso_code, year, events, fatalities
acled_combined = acled_combined[["country", "iso_code", "year", "events", "fatalities"]]

# === E-GOV DATA ===

egov = pd.read_csv(os.path.join(raw_data_path, "egov_un_data.csv"))
egov = egov.rename(columns={
    "Country Name": "country",
    "E-Participation Index": "e_participation_index",
    "Survey Year": "year"
})

# Keep only relevant columns
egov = egov[["country", "e_participation_index", "year"]]

# Clean country names before merging
egov["country"] = (
    egov["country"]
    .str.strip()
    .str.replace('"', '')
    .str.replace(r'\s+', ' ', regex=True)
)

# Manual corrections
manual_corrections = {
    "Iran (Islamic Republic of)": "Iran",
    "Viet Nam": "Vietnam",
    "Türkiye": "Turkey",
    "United States of America": "United States",
    "Republic of Korea": "South Korea",
}
egov["country"] = egov["country"].replace(manual_corrections)

# Now merge after cleaning
egov = egov.merge(iso_df, on="country", how="left")


# Reorder columns so that iso_code is right after country
cols = egov.columns.tolist()
cols.remove("iso_code")
cols.insert(cols.index("country") + 1, "iso_code")
egov = egov[cols]

# === FREEDOM ON THE NET DATA ===
freedom = pd.read_excel(
    os.path.join(raw_data_path, "freedom_net_country_score_data.xlsx"),
    skiprows=1,
    engine="openpyxl"  # Make sure openpyxl is installed
)
freedom = freedom.rename(columns={
    "Country": "country",
    "Edition": "year",
    "Total": "media_freedom_score"
})
freedom = freedom[["country", "year", "media_freedom_score"]]
freedom = freedom.merge(iso_df, on="country", how="left")
freedom = freedom.dropna(subset=["iso_code"])

# Reorder columns so that iso_code is right after country
cols = freedom.columns.tolist()
cols.remove("iso_code")
cols.insert(cols.index("country") + 1, "iso_code")
freedom = freedom[cols]

# === POLARISATION SCORES ===
polarisation = pd.read_csv(os.path.join(polarisation_path, "merged_polarization_scores.csv"))

# Clean column names
polarisation = polarisation.rename(columns={"country": "iso_code"})
polarisation = polarisation.rename(columns={"country_full": "country"})

# Move 'country' to the first column
cols = ['country'] + [col for col in polarisation.columns if col != 'country']
polarisation = polarisation[cols]

# === 🆕 WPF SCORES ===

wpf = pd.read_csv(os.path.join(wpf_path, "wpf_scores.csv"))

# Merge ISO codes using the iso_df mapping
wpf = wpf.merge(iso_df, on="country", how="left")

# Drop rows with missing ISO codes
wpf = wpf.dropna(subset=["iso_code"])

# Reorder columns so that iso_code is right after country
cols = wpf.columns.tolist()
cols.remove("iso_code")
cols.insert(cols.index("country") + 1, "iso_code")
wpf = wpf[cols]


# === Tertiary Enrollment Education ===

education = pd.read_csv(os.path.join(raw_data_path, "tertiary_enrollment_wb.csv"))

# Clean column names
education = education.rename(columns={"Country Name": "country", "Country Code": "iso_code"})

# Drop unused columns
education = education.drop(columns=["Series Name", "Series Code"])

# Melt the DataFrame to long format
education = education.melt(
    id_vars=["country", "iso_code"],
    var_name="year",
    value_name="enrollment_percentage"
)

# Clean up the year column: extract only the year (e.g., from "2020 [YR2020]" to 2020)
education["year"] = education["year"].str.extract(r'(\d{4})').astype("Int64")

# Create a list of valid country names from the values of the JSON
valid_countries = list(iso_mapping.values())

# Filter to only include countries in the ISO mapping
education = education[education["country"].isin(valid_countries)]

# Final column order (optional but clean)
education = education[["country", "iso_code", "year", "enrollment_percentage"]]

# === FINAL MERGE ===
df = pd.merge(acled_combined, egov, on=["country", "iso_code", "year"], how="outer")
df = pd.merge(df, freedom, on=["country", "iso_code", "year"], how="outer")
df = pd.merge(df, polarisation, on=["country", "iso_code", "year"], how="outer")
df = pd.merge(df, wpf, on=["country", "iso_code", "year"], how="outer")  
df = pd.merge(df, education, on=["country", "iso_code", "year"], how="outer")

# Reorder columns: put 'iso_code' and 'year' together, followed by other columns
columns_order = ['country', 'iso_code', 'year'] + [col for col in df.columns if col not in ['country', 'iso_code', 'year']]
df = df[columns_order]

# === SAVE OUTPUT ===
output_file = os.path.join(output_path, "misinformation_vulnerability_risk_index.csv")
df.to_csv(output_file, index=False)

## Building the Misinformation Vulnerability Index

In [127]:
# Clean and filter year
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df = df[df['year'].isin([2011, 2013] + list(range(2018, 2025)))]

# Drop columns mean,ci_low,ci_high
df = df.drop(columns=['mean', 'ci_low', 'ci_high'], errors='ignore')

# Create output path
final_output_path = "../data/reports/misinformation_index"
os.makedirs(final_output_path, exist_ok=True)

# Save final DataFrame
df.to_csv(os.path.join(final_output_path, "data_misinformation_index.csv"), index=False)

print("✅ File saved as data_misinformation_index.csv")

✅ File saved as data_misinformation_index.csv
